In [1]:
import sympy as sp
from sympy import sin, cos, pi
from sympy.interactive import printing
import pickle
import numpy as np
import scipy as sc
import scipy.interpolate
from scipy.integrate import odeint
import matplotlib.pyplot as pl

import symbtools as st
import symbtools.modeltools as mt
import symbtools.noncommutativetools as nct

In [2]:
t = sp.Symbol('t')
Np = 1
Nq = 1
n = Np + Nq
pp = st.symb_vector("p1:{0}".format(Np+1))
qq = st.symb_vector("q1:{0}".format(Nq+1))
aa = st.symb_vector("a1:{0}".format(Nq+1))
ww = st.symb_vector("w1:{0}".format(Nq+1))

ttheta = st.row_stack(pp, qq) ##:T
tthetad = st.time_deriv(ttheta, ttheta) ##:T
tthetadd = st.time_deriv(ttheta, ttheta, order=2) ##:T
st.make_global(ttheta, tthetad)

In [3]:
params = sp.symbols('m1, m2, s2, g') # m1 wagen, m2 pole
st.make_global(params)

tau1 = sp.Symbol("tau1")
k = sp.var("k") # k = 4/3
l = sp.var("l")

In [4]:
#Einheitsvektoren
ex = sp.Matrix([1,0])
ey = sp.Matrix([0,1])


# Koordinaten der Schwerpunkte und Gelenke
S1 = ex*q1 # Schwerpunkt Wagen
G2 = S1 # Pendel-Gelenk

# Schwerpunkt des Pendels (Pendel zeigt für kleine Winkel nach #! OBEN)
S2 = G2 +sp.Matrix([l/2 * sin(p1), l/2*cos(p1)]) #- mt.Rz(p1)*(ey)*s2/2 #! paper: pole has length 2*s2

# Zeitableitungen der Schwerpunktskoordinaten
Sd1, Sd2  = st.col_split(st.time_deriv(st.col_stack(S1, S2), ttheta)) ##

In [5]:
S2

Matrix([
[l*sin(p1)/2 + q1],
[     l*cos(p1)/2]])

In [6]:
Sd2

Matrix([
[l*pdot1*cos(p1)/2 + qdot1],
[       -l*pdot1*sin(p1)/2]])

In [7]:
# Energie
# T_rot = 0 # (Punktmassenmodell)
# J = k*m2*s2**2
J = sp.var("J")
# T_rot = 0.5 * (J * (Sd2[0]**2 + Sd2[1]**2) / s2**2) # (Verteilte-Masse)
T_rot = 0#0.5 * (J * tthetad[0]**2) # (Verteilte-Masse)
T_trans = ( m1*Sd1.T*Sd1  +  m2*Sd2.T*Sd2 )/2

T = T_rot + T_trans[0]

V = m2*g*S2[1]

In [8]:
tthetad

Matrix([
[pdot1],
[qdot1]])

In [9]:
sp.simplify(T_rot)

0

In [10]:
T_trans

Matrix([[l**2*m2*pdot1**2*sin(p1)**2/8 + m1*qdot1**2/2 + m2*(l*pdot1*cos(p1)/2 + qdot1)**2/2]])

In [11]:
L = T - V
sp.simplify(L)

-g*l*m2*cos(p1)/2 + l**2*m2*pdot1**2/8 + l*m2*pdot1*qdot1*cos(p1)/2 + m1*qdot1**2/2 + m2*qdot1**2/2

In [12]:
# friction
u = sp.var("u")

In [13]:
mod = mt.generate_symbolic_model(T, V, ttheta, [0, tau1])

In [14]:
mod.eqns

Matrix([
[                       l*m2*(-2*g*sin(p1) + l*pddot1 + 2*qddot1*cos(p1))/4],
[m1*qddot1 + m2*(l*pddot1*cos(p1) - l*pdot1**2*sin(p1) + 2*qddot1)/2 - tau1]])

In [15]:
mod.MM.simplify()
mod.MM

Matrix([
[     l**2*m2/4, l*m2*cos(p1)/2],
[l*m2*cos(p1)/2,        m1 + m2]])

In [16]:
mod.calc_coll_part_lin_state_eq(simplify=True)

In [17]:
mod.calc_state_eq(simplify=True)

In [18]:
sp.cancel(sp.together(mod.state_eq))

Matrix([
[                                                                                                      pdot1],
[                                                                                                      qdot1],
[(2*g*m1*sin(p1) + 2*g*m2*sin(p1) - l*m2*pdot1**2*sin(p1)*cos(p1) - 2*tau1*cos(p1))/(l*m1 + l*m2*sin(p1)**2)],
[                        (-2*g*m2*sin(p1)*cos(p1) + l*m2*pdot1**2*sin(p1) + 2*tau1)/(2*m1 + 2*m2*sin(p1)**2)]])

In [19]:
print(sp.cancel(sp.together(mod.state_eq)))

Matrix([[pdot1], [qdot1], [(2*g*m1*sin(p1) + 2*g*m2*sin(p1) - l*m2*pdot1**2*sin(p1)*cos(p1) - 2*tau1*cos(p1))/(l*m1 + l*m2*sin(p1)**2)], [(-2*g*m2*sin(p1)*cos(p1) + l*m2*pdot1**2*sin(p1) + 2*tau1)/(2*m1 + 2*m2*sin(p1)**2)]])


Mit Reibung

In [20]:
mod2 = mt.generate_symbolic_model(T, V, ttheta, [-u*tthetad[0], tau1])
mod2.calc_state_eq(simplify=True)
sp.cancel(sp.together(mod2.state_eq))

Matrix([
[                                                                                                                                                                     pdot1],
[                                                                                                                                                                     qdot1],
[(2*g*l*m1*m2*sin(p1) + 2*g*l*m2**2*sin(p1) - l**2*m2**2*pdot1**2*sin(p1)*cos(p1) - 2*l*m2*tau1*cos(p1) - 4*m1*pdot1*u - 4*m2*pdot1*u)/(l**2*m1*m2 + l**2*m2**2*sin(p1)**2)],
[                                                        (-2*g*l*m2*sin(p1)*cos(p1) + l**2*m2*pdot1**2*sin(p1) + 2*l*tau1 + 4*pdot1*u*cos(p1))/(2*l*m1 + 2*l*m2*sin(p1)**2)]])

In [21]:
print(sp.cancel(sp.together(mod2.state_eq)))

Matrix([[pdot1], [qdot1], [(2*g*l*m1*m2*sin(p1) + 2*g*l*m2**2*sin(p1) - l**2*m2**2*pdot1**2*sin(p1)*cos(p1) - 2*l*m2*tau1*cos(p1) - 4*m1*pdot1*u - 4*m2*pdot1*u)/(l**2*m1*m2 + l**2*m2**2*sin(p1)**2)], [(-2*g*l*m2*sin(p1)*cos(p1) + l**2*m2*pdot1**2*sin(p1) + 2*l*tau1 + 4*pdot1*u*cos(p1))/(2*l*m1 + 2*l*m2*sin(p1)**2)]])


In [22]:
xx = mod.x
f = mod.ff
G = mod.gg
g1, = st.col_split(G)

In [23]:
f

Matrix([
[        pdot1],
[        qdot1],
[2*g*sin(p1)/l],
[            0]])

In [24]:
G

Matrix([
[           0],
[           0],
[-2*cos(p1)/l],
[           1]])

In [25]:
eqp1 = sp.Matrix([0, 0, 0, 0 ]) ##:T
eqp2 = sp.Matrix([sp.pi, 0, 0, 0]) ##:T

# Probe:
assert f.subz(xx, eqp1) == sp.zeros(4,1)
assert f.subz(xx, eqp2) == sp.zeros(4,1)

In [41]:
# Parameterwerte
parameter_values = [(g, 9.81), (l, 1), (k, 4/3)]
replm =  parameter_values + list(zip(xx, eqp2))

In [42]:
mod.x


Matrix([
[   p1],
[   q1],
[pdot1],
[qdot1]])

In [43]:
A1 = f.jacobian(xx).subs(replm)
A1

Matrix([
[     0, 0, 1, 0],
[     0, 0, 0, 1],
[-19.62, 0, 0, 0],
[     0, 0, 0, 0]])

In [44]:
b1 = G.subs(replm)
b1

Matrix([
[0],
[0],
[2],
[1]])

In [48]:
a = 1.2
b = 3.4

# poles = np.r_[-1.5+a*1j, -1.5-a*1j, -1.3 + b*1j, -1.3 - b*1j] ##:
poles = np.linspace(-5, -8, 4)
# k1 = st.siso_place(A1, b1, poles)
# print(k1)
# A1-b1@k1
import control as ctrl
F = ctrl.place(np.array(A1, dtype=float), np.array(b1, dtype=float), poles)
print("F", F)
np.linalg.eigvals(np.array(A1, dtype=float)- np.array(b1,dtype=float)@F)


F [[ 72.87654434  85.62691131 -14.16615698  54.33231397]]


array([-8., -7., -6., -5.])